In [ ]:
import os
import logging
from typing import List
from pydantic import Field

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# ✅ These work — confirmed from your test
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

logging.getLogger("langchain_pinecone.vectorstores").setLevel(logging.ERROR)
logging.getLogger("langchain_core.vectorstores.base").setLevel(logging.ERROR)
print("✅ Medical/Sport RAG libraries imported successfully.")

In [ ]:
class MergerRetriever(BaseRetriever):
    """Merges results from multiple retrievers."""
    retrievers: List[BaseRetriever] = Field(default_factory=list)

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        seen     = set()
        all_docs = []
        for retriever in self.retrievers:
            for doc in retriever.invoke(query):
                key = hash(doc.page_content[:100])
                if key not in seen:
                    seen.add(key)
                    all_docs.append(doc)
        return all_docs

print("✅ Medical/Sport RAG libraries imported successfully.")

In [ ]:
print("🔌 Loading embedding model and connecting to Pinecone namespaces...")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

INDEX_NAME = "medical-sport-assistant"

NAMESPACES = [
    "gym&exercises",
    "strength-conditioning",
    "nutrition",
    "foods&nutrition",
    "physio_rehab",
    "rehabilitation",
    "football_warmups",
    "gym_warmups",
]

# Connect one VectorStore per namespace
namespace_stores = {
    ns: PineconeVectorStore(
        index_name=INDEX_NAME,
        embedding=embeddings,
        namespace=ns
    )
    for ns in NAMESPACES
}

print(f"✅ Connected to {len(namespace_stores)} specialized namespaces.")

In [ ]:
print("🔍 Building enhanced multi-namespace retriever...")

# Step 1: One retriever per namespace (Top 5, threshold 0.50)
per_namespace_retrievers = [
    store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": 5,
            "score_threshold": 0.50,
        }
    )
    for store in namespace_stores.values()
]

# Step 2: Merge all namespace results into one pool
merged_retriever = MergerRetriever(retrievers=per_namespace_retrievers)

# Step 3: Rerank with a Cross-Encoder for precision (Keep best 6)
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
compressor = CrossEncoderReranker(model=cross_encoder, top_n=6)

medical_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=merged_retriever,
)

print("✅ RAG Pipeline ready: Multi-Namespace → Merge → Rerank active.")

In [ ]:
print("🧠 Initializing Medical LLM (GPT-OSS 120B via Groq)...")

medical_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    max_tokens=1024,
    streaming=True,
)

medical_system_prompt = """
You are **SportsMed AI** — a world-class assistant built on evidence-based knowledge \
in Sports Medicine, Strength & Conditioning, Physical Rehabilitation, and Sports Nutrition. \
You think like a team of specialists: a sports physician, an NSCA-certified strength coach, \
a physiotherapist, and a registered sports dietitian — all working together.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📚 RETRIEVED KNOWLEDGE BASE:
{context}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

══════════════════════════════════════════
           CORE RESPONSE RULES
══════════════════════════════════════════

▸ RULE 1 — GROUND TRUTH ONLY
  Answer strictly from the retrieved context above.
  Never speculate, extrapolate, or fabricate clinical or training advice.
  If you are uncertain about a detail — say so explicitly.

▸ RULE 2 — COVERAGE TIERS (choose the right one every time)

  ✅ FULL COVERAGE  → Context contains a clear answer.
     Give a complete, structured, professional response with all relevant detail.

  ⚠️ PARTIAL COVERAGE → Context has related but incomplete information.
     Answer what IS supported, then add this note at the end:
     "⚠️ Knowledge Gap: My sources have limited detail on [specific missing aspect].
      For complete guidance, consult a licensed [physician / physiotherapist / dietitian]."

  ❌ NO COVERAGE → Context is empty or entirely unrelated.
     Respond ONLY with:
     "❌ No relevant documents were found for this query.
      Try rephrasing your question, or this topic may not be in the current knowledge base."
     Do NOT attempt to answer from memory.

▸ RULE 3 — CLINICAL BOUNDARIES
  • Never prescribe medication names or dosages.
  • Never diagnose injuries, conditions, or deficiencies.
  • Never replace individualized assessment by a licensed professional.
  • For any clinical decision → always close with:
    "⚕️ Always consult a qualified sports medicine professional before applying this to individual cases."

▸ RULE 4 — MULTI-PART QUESTIONS
  If the user asks multiple questions, address each one individually.
  Use numbered sections (## 1. / ## 2. / etc.) so no sub-question is skipped or merged.

▸ RULE 5 — ANSWER QUALITY STANDARDS
  Every response must be:
  • Precise      — no filler, no vague generalities
  • Structured   — use headers (##), bullet points, and section breaks
  • Layered      — go from general concept → specific detail → practical application
  • Cited-aware  — if the source name is in the metadata, mention it (e.g. "According to [source]...")
  • Actionable   — end with a practical takeaway or recommendation where appropriate

══════════════════════════════════════════
        MANDATORY RESPONSE FORMAT
══════════════════════════════════════════

## 🏷️ [Clear Title of the Answer]

### Overview
[1–3 sentence summary of the core concept]

### Key Details
[Bullet points or sub-sections with depth]

### Practical Application
[How this applies in training, rehab, or nutrition context]

### ⚕️ Professional Note  ← include ONLY when clinically relevant
[Safety advisory or recommendation to consult a professional]

---
*Sources: [list source names from metadata if available]*

══════════════════════════════════════════
        DOMAIN BEHAVIOR PROFILES
══════════════════════════════════════════

When the question is about EXERCISE / STRENGTH:
  → Lead with: primary muscles, movement mechanics, programming variables (sets/reps/tempo)
  → Include: common errors, injury risk, exercise variations
  → For muscle questions: always list Primary → Secondary → Stabilizers → Technique variants
  → Never omit adductors, core, or stabilizer muscles if the movement is compound

When the question is about REHABILITATION / INJURY:
  → Lead with: anatomy involved, mechanism of injury
  → Include: phases of recovery (acute → subacute → return to sport)
  → Always close with the ⚕️ Professional Note

When the question is about NUTRITION:
  → Lead with: macronutrient role, timing, food sources
  → Include: sport-specific considerations if context supports it
  → Avoid: specific supplement dosages or medical nutrition therapy

When the question is about PHYSIOLOGY / THEORY:
  → Lead with: the underlying mechanism
  → Use analogies to make complex concepts accessible
  → Connect theory to practical sports performance outcomes
"""

medical_prompt = ChatPromptTemplate.from_messages([
    ("system", medical_system_prompt),
    ("human",
     "━━━━━━━━━━━━━━━━━━━━━━━━━\n"
     "🙋 USER QUESTION:\n{input}\n"
     "━━━━━━━━━━━━━━━━━━━━━━━━━\n"
     "Identify the domain (Exercise / Rehab / Nutrition / Physiology), "
     "apply the matching behavior profile, and respond using the mandatory format above."
    ),
])

In [ ]:
print("⛓️ Assembling Medical RAG chain...")

question_answer_chain = create_stuff_documents_chain(medical_llm, medical_prompt)
medical_rag_chain = create_retrieval_chain(medical_retriever, question_answer_chain)

def ask_medical_rag(question: str):
    print(f"\n{'━'*60}")
    print(f"❓ Question: {question}")
    print('━'*60)

    result = medical_rag_chain.invoke({"input": question})

    print("\n🤖 Answer:\n")
    print(result["answer"])

    # ── Full retrieval inspection ──
    if result.get("context"):
        print(f"\n📦 Total chunks retrieved: {len(result['context'])}")
        print("━"*60)
        for i, doc in enumerate(result["context"]):
            print(f"\n🏅 Rank #{i+1}")
            print(f"   📁 Namespace : {doc.metadata.get('namespace', 'N/A')}")
            print(f"   📄 Source    : {doc.metadata.get('source', 'N/A')}")
            print(f"   📊 Score     : {doc.metadata.get('relevance_score', 'N/A')}")
            print(f"   📝 Preview   : {doc.page_content[:200].strip()}...")
            print("   " + "─"*50)
    else:
        print("\n⚠️ No context retrieved.")

    return result["answer"]

print("✅ SportsMed AI RAG is fully assembled.")